# Patch extraction walkthrough

Cut full-resolution image/mask pairs from a folder into overlapping 512×512 training patches.

Expected layout:

```
raw_dir/
├── images/   sample_001.png  (or sample_001_ch0.png, sample_001_ch1.png, …)
└── masks/    sample_001.png  (same stem; or sample_001_mask.png)
```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import cv2

from segmentation_tools_pt.dataset_build import build_patch_dataset, discover_samples
from segmentation_tools_pt.patches import PatchConfig

In [ ]:
# ── point at your data ───────────────────────────────────────────────────────
RAW_DIR = Path('/path/to/dataset/segmentation/example_lot')
PATCH_DIR = RAW_DIR.parent / f'{RAW_DIR.name}_patches_512'

N_CHANNELS = 1          # 1 = images/{id}.png ; 2 = images/{id}_ch0.png + _ch1.png
MASK_SUFFIX = ''        # '' when mask stem matches image stem; '_mask' for {id}_mask.png
NUM_CLASSES = 1         # binary segmentation
FIT_RECTS = True        # snap noisy mask blobs to rotated rectangles before patching

# patch_size=512 with stride=256 → 50% overlap between neighbouring patches
PATCH_SIZE = 512
STRIDE = 256              # overlap = patch_size - stride = 256 px

cfg = PatchConfig(
    patch_size=PATCH_SIZE,
    stride=STRIDE,
    min_fg_pixels=50,       # patches with fewer fg pixels are 'empty'
    max_empty_fraction=0.10,  # keep ~10% of empty patches for hard-negative mining
    val_fraction=0.20,
    seed=42,
)

print(f'raw:    {RAW_DIR}')
print(f'output: {PATCH_DIR}')
print(f'overlap: {PATCH_SIZE - STRIDE}px  ({100 * (PATCH_SIZE - STRIDE) / PATCH_SIZE:.0f}%)')

In [ ]:
# preview what will be patched
sample_ids = discover_samples(
    str(RAW_DIR),
    n_channels=N_CHANNELS,
    mask_suffix=MASK_SUFFIX,
)
print(f'{len(sample_ids)} source image(s)')
print('first few:', sample_ids[:5])

In [ ]:
# extract patches → PATCH_DIR/train/ and PATCH_DIR/val/
meta = build_patch_dataset(
    source_dir=str(RAW_DIR),
    output_dir=str(PATCH_DIR),
    cfg=cfg,
    num_classes=NUM_CLASSES,
    n_channels=N_CHANNELS,
    n_jobs=4,               # parallel CPU workers; set 1 for debugging
    fit_rects=FIT_RECTS,
    mask_suffix=MASK_SUFFIX,
)

n_train = len(meta['train'])
n_val = len(meta['val'])
n_fg = sum(1 for m in meta['train'] if not m['is_empty'])
print(f'train patches: {n_train}  ({n_fg} with foreground)')
print(f'val patches:   {n_val}')
print(f'metadata:      {PATCH_DIR / "metadata.json"}')

In [ ]:
# quick visual check — one foreground patch from the train split
fg_patches = [m for m in meta['train'] if not m['is_empty']]
assert fg_patches, 'no foreground patches — lower min_fg_pixels or check masks'

m = fg_patches[0]
pid = m['sample_id']
# Single-channel patches share the same base name in images/ and masks/.
# Multi-channel images use _ch{i}; legacy caches may use _mask for masks.
img_path = PATCH_DIR / 'train' / 'images' / f'{pid}.png'
if not img_path.exists():
    img_path = PATCH_DIR / 'train' / 'images' / f'{pid}_ch0.png'
msk_path = PATCH_DIR / 'train' / 'masks' / f'{pid}.png'
if not msk_path.exists():
    msk_path = PATCH_DIR / 'train' / 'masks' / f'{pid}_mask.png'

img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
msk = cv2.imread(str(msk_path), cv2.IMREAD_GRAYSCALE)

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(img, cmap='gray'); ax[0].set_title('patch image'); ax[0].axis('off')
ax[1].imshow(msk, cmap='gray'); ax[1].set_title('patch mask'); ax[1].axis('off')
ax[2].imshow(img, cmap='gray')
ax[2].imshow((msk > 0).astype(float), alpha=0.4, cmap='Reds')
ax[2].set_title(f"{pid}\nfg={m['fg_pixels']} px"); ax[2].axis('off')
plt.tight_layout()